### INCREMENTAL DATA LOAD

In [0]:
%sql
create volume dev.bronze.landing;

In [0]:
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/input")

In [0]:
# Copy retail invoice data from databricks-datasets

dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv", "/Volumes/dev/bronze/landing/input")

In [0]:

## Copy 2nd retail invoice data from databricks-datasets

dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv", "/Volumes/dev/bronze/landing/input")

In [0]:
## Copy 3rd retail invoice data from databricks-datasets

dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv", "/Volumes/dev/bronze/landing/input")

In [0]:
%sql
-- create placeholder table dev.bronze.invoice_cp
create table dev.bronze.invoice_cp;

In [0]:
%sql
copy into dev.bronze.invoice_cp
from '/Volumes/dev/bronze/landing/input'
fileformat = CSV
pattern = '*.csv'
format_options ('header' = 'true', 'mergeschema' = 'true')
copy_options ('mergeSchema' = 'true');
    
--select * from dev.bronze.invoice_cp


In [0]:
%sql

--describe table
describe extended dev.bronze.invoice_cp

In [0]:
%sql
--- create table with only 3 cols

create table dev.bronze.invoice_cp_alt(
  invoiceNo string,
  StockCode string,
  Quantity double,
  _insert_date timestamp
)

In [0]:
%sql

-- custom col inclusion and current timestamp
copy into dev.bronze.invoice_cp_alt
from (
  select invoiceNo, StockCode, cast(Quantity as double) Quantity, current_timestamp() as _insert_date
  from '/Volumes/dev/bronze/landing/input')
  fileformat = CSV
  pattern = '*.csv'
  format_options ('header' = 'true', 'mergeschema' = 'true')
;